# MoE-GAL-MAD Model Implementation (VERSION 4)
- Stride/Shift = 16 timesteps
- Window size = 24 timesteps
- Label pooling = Max over timesteps
- Architecture: Mixture of Experts with GAL-MAD Experts (1 GAT + 1 BiLSTM)
- Router: Per-window gating (Input 6336)

In [ ]:
!pip install torch torchvision torchaudio
!pip install torch_geometric shap pandas scikit-learn numpy

## 1. Constants & Definitions

In [ ]:

services = ['payment', 'shipping', 'redis', 'mongodb', 'dispatch', 'rabbitmq', 'user', 'mysql', 'catalogue', 'ratings', 'web', 'cart']
anomalies = ['rt-delay-catalogue',
 'packetloss-user',
 'low-bandwidth-user',
 'high-cpu-dispatch',
 'high-latency-user-2',
 'high-load-1500',
 'out-of-order-packets-user-2',
 'high-latency-user',
 'service-down-payment',
 'out-of-order-packets-user',
 'packetloss-user-2',
 'high-fileIO-payment',
 'memory-leak-cart',
  'low-bandwidth-user-2']


cumulative_cols = ['container_cpu_system_seconds_total',
'container_cpu_usage_seconds_total',
'container_cpu_user_seconds_total',
'container_network_receive_bytes_total',
'container_network_receive_errors_total',
'container_network_receive_packets_dropped_total',
'container_network_receive_packets_total',
'container_network_transmit_bytes_total',
'container_network_transmit_errors_total'	,
'container_network_transmit_packets_dropped_total',
'container_network_transmit_packets_total',
'container_fs_io_time_seconds_total',
'container_memory_failures_total',
'container_memory_failcnt',
'container_fs_write_seconds_total']

other_cols = ['container_fs_usage_bytes',
'container_memory_rss',
'container_memory_usage_bytes',
'container_memory_working_set_bytes']

cols = other_cols + cumulative_cols

containers = ['payment', 'shipping', 'redis', 'mongo', 'dispatch', 'rabbitmq', 'user', 'mysql', 'catalogue', 'ratings', 'web', 'cart']
metrics = ['rt_ratings_put_PDO_count', 'rt_ratings_put_PDO_sum', 'rt_payment_delete_cart_count', 'rt_payment_delete_cart_sum', 'rt_cart_get_catalogue_count', 'rt_cart_get_catalogue_sum', 'rt_ratings_get_catalogue_count', 'rt_ratings_get_catalogue_sum', 'rt_catalogue_get_mongo_categories_count', 'rt_catalogue_get_mongo_categories_sum', 'rt_catalogue_get_mongo_products_count', 'rt_catalogue_get_mongo_products_sum', 'rt_catalogue_get_mongo_productscat_count', 'rt_catalogue_get_mongo_productscat_sum', 'rt_catalogue_get_mongo_productsku_count', 'rt_catalogue_get_mongo_productsku_sum', 'rt_catalogue_get_mongo_search_count', 'rt_catalogue_get_mongo_search_sum', 'rt_user_get_mongo_checkid_count', 'rt_user_get_mongo_checkid_sum', 'rt_user_get_mongo_history_count', 'rt_user_get_mongo_history_sum', 'rt_user_get_mongo_users_count', 'rt_user_get_mongo_users_sum', 'rt_user_post_mongo_login_count', 'rt_user_post_mongo_login_sum', 'rt_user_post_mongo_order_count', 'rt_user_post_mongo_order_sum', 'rt_user_post_mongo_register_count', 'rt_user_post_mongo_register_sum', 'rt_web_post_payment_count', 'rt_web_post_payment_sum', 'rt_dispatch_get_rabbitmq_count', 'rt_dispatch_get_rabbitmq_sum', 'rt_web_get_ratings_count', 'rt_web_get_ratings_sum', 'rt_cart_delete_redis_count', 'rt_cart_delete_redis_sum', 'rt_cart_get_redis_count', 'rt_cart_get_redis_sum', 'rt_cart_post_redis_count', 'rt_cart_post_redis_sum', 'rt_user_get_redis_count', 'rt_user_get_redis_sum', 'rt_web_get_shipping_calcid_seconds_count', 'rt_web_get_shipping_calcid_seconds_sum', 'rt_web_get_shipping_code_seconds_count', 'rt_web_get_shipping_code_seconds_sum', 'rt_web_get_shipping_postconfirm_seconds_count', 'rt_web_get_shipping_postconfirm_seconds_sum', 'rt_payment_get_user_count', 'rt_payment_get_user_sum', 'rt_payment_post_user_count', 'rt_payment_post_user_sum']


## 2. Dataset and Preprocessing

In [ ]:
import torch
from torch.utils.data import Dataset
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

class RSAnomicDataset(Dataset):
    def __init__(self, data_path, seq_len=24, stride=16, is_train=True, scaler=None):
        self.seq_len = seq_len
        self.stride = stride
        self.is_train = is_train
        
        raw_data = torch.load(data_path) 
        L = raw_data.shape[0]
        num_nodes = raw_data.shape[1]
        
        if is_train:
            features = raw_data
            self.labels = None
        else:
            features = raw_data[:, :, :20]
            self.labels = raw_data[:, :, 20].numpy()
            
        features_np = features.numpy()
        
        ma_features = np.zeros((L, num_nodes, 2))
        for n in range(num_nodes):
            rt_series = pd.Series(features_np[:, n, 19])
            ma_24 = rt_series.rolling(window=24, min_periods=1).mean()
            ma_features[:, n, 0] = rt_series - ma_24
            ma_240 = rt_series.rolling(window=240, min_periods=1).mean()
            ma_features[:, n, 1] = rt_series - ma_240
            
        features_np = np.concatenate([features_np, ma_features], axis=-1)
        features_flat = features_np.reshape(-1, 22)
        
        if is_train:
            if scaler is None:
                self.scaler = StandardScaler()
                features_flat = self.scaler.fit_transform(features_flat)
            else:
                self.scaler = scaler
                features_flat = self.scaler.transform(features_flat)
        else:
            self.scaler = scaler
            if self.scaler is None:
                raise ValueError("Scaler must be provided for test data")
            features_flat = self.scaler.transform(features_flat)
            
        self.data = features_flat.reshape(L, num_nodes, 22)
        self.data = torch.tensor(self.data, dtype=torch.float32)
        
        if self.labels is not None:
            self.labels = torch.tensor(self.labels, dtype=torch.float32)
            
        self.num_samples = (L - self.seq_len) // self.stride + 1

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        start_idx = idx * self.stride
        x = self.data[start_idx : start_idx + self.seq_len]
        
        if self.labels is not None:
            window_labels = self.labels[start_idx : start_idx + self.seq_len]
            # Nếu chỉ cần 1 timestep lỗi trong 24 timestep, coi như lỗi cả cửa sổ đó
            # max(dim=0)[0] sẽ lấy giá trị lớn nhất theo chiều thời gian (0 hoặc 1)
            y = window_labels.max(dim=0)[0]
            return x, y
        
        return x


## 2.5. Visualize Raw Data
Biểu đồ dưới đây trực quan hóa 20 features gốc (trước khi tính Moving Average) của vi dịch vụ `shipping` (node index = 1) trên cả tập Train và Test.

In [ ]:
import torch
import matplotlib.pyplot as plt
import os

# Cập nhật đường dẫn file cho phù hợp với môi trường Colab/Local của bạn
train_path = 'train/train.pt' if os.path.exists('train/train.pt') else 'E:/VHT/VDT/AD/train/train.pt'
test_path = 'test/test5.pt' if os.path.exists('test/test5.pt') else 'E:/VHT/VDT/AD/test/test5.pt'

try:
    train_raw = torch.load(train_path)
    test_raw = torch.load(test_path)

    # Chọn microservice 'shipping' (index 1)
    node_idx = 1
    ms_name = 'shipping'

    # Lấy 20 features gốc (index từ 0 đến 19)
    train_feats = train_raw[:, node_idx, :20].numpy()
    test_feats = test_raw[:, node_idx, :20].numpy()

    fig, axs = plt.subplots(2, 1, figsize=(15, 10))

    # Plot Train
    for i in range(20):
        axs[0].plot(train_feats[:, i], label=f'Feature {i}', alpha=0.7)
    axs[0].set_title(f'Train Data - 20 Raw Features ({ms_name})')
    axs[0].set_xlabel('Timesteps')
    axs[0].set_ylabel('Value')
    
    # Plot Test
    for i in range(20):
        axs[1].plot(test_feats[:, i], label=f'Feature {i}', alpha=0.7)
    axs[1].set_title(f'Test Data - 20 Raw Features ({ms_name})')
    axs[1].set_xlabel('Timesteps')
    axs[1].set_ylabel('Value')

    plt.tight_layout()
    plt.show()
except FileNotFoundError:
    print("File không tồn tại. Vui lòng kiểm tra lại đường dẫn file train.pt và test5.pt")


## 3. MoE-GAL-MAD Architecture

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv

# Define edges
source_nodes = [10, 10, 10, 10, 10, 10, 11, 11, 6, 6, 8, 0, 0, 0, 5, 1, 9]
target_nodes = [11, 8, 9, 1, 0, 6, 2, 8, 2, 3, 3, 6, 5, 11, 4, 7, 7]
EDGE_INDEX = torch.tensor([source_nodes, target_nodes], dtype=torch.long)

class GALMAD_Encoder(nn.Module):
    def __init__(self, in_channels=22, hidden_dim=32):
        super().__init__()
        self.gat = GATConv(in_channels, hidden_dim)
        self.lstm = nn.LSTM(
            input_size=hidden_dim, 
            hidden_size=hidden_dim // 2, 
            num_layers=1, 
            batch_first=True, 
            bidirectional=True
        )
        self.linear = nn.Linear(hidden_dim, 1) # d_z = 1
        
    def forward(self, x, edge_index):
        batch_size, seq_len, num_nodes, in_channels = x.shape
        num_graphs = batch_size * seq_len
        
        x_flat = x.view(num_graphs * num_nodes, in_channels)
        offsets = torch.arange(num_graphs, device=x.device) * num_nodes
        edge_index_batched = edge_index.unsqueeze(2) + offsets.view(1, 1, -1)
        edge_index_batched = edge_index_batched.view(2, -1)
        
        # GAT (Spatial)
        x_gat = F.relu(self.gat(x_flat, edge_index_batched)) 
        
        # LSTM (Temporal)
        x_gat = x_gat.view(batch_size, seq_len, num_nodes, -1)
        x_gat = x_gat.transpose(1, 2).contiguous() 
        x_lstm_in = x_gat.view(batch_size * num_nodes, seq_len, -1)
        
        out_lstm, (hn, cn) = self.lstm(x_lstm_in)
        final_hidden = torch.cat([hn[-2], hn[-1]], dim=-1) 
        
        z = self.linear(final_hidden) 
        z = z.view(batch_size, num_nodes, 1)
        return z

class GALMAD_Decoder(nn.Module):
    def __init__(self, out_channels=22, hidden_dim=32):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=1, 
            hidden_size=hidden_dim // 2, 
            num_layers=1, 
            batch_first=True, 
            bidirectional=True
        )
        self.linear = nn.Linear(hidden_dim, hidden_dim)
        self.gat = GATConv(hidden_dim, out_channels)
        
    def forward(self, z, seq_len, edge_index):
        batch_size, num_nodes, _ = z.shape
        z_lstm_in = z.view(batch_size * num_nodes, 1)
        z_lstm_in = z_lstm_in.unsqueeze(1).repeat(1, seq_len, 1)
        
        out_lstm, _ = self.lstm(z_lstm_in)
        x_gat_in = F.relu(self.linear(out_lstm)) 
        
        x_gat_in = x_gat_in.view(batch_size, num_nodes, seq_len, -1)
        x_gat_in = x_gat_in.transpose(1, 2).contiguous().view(batch_size * seq_len * num_nodes, -1)
        
        num_graphs = batch_size * seq_len
        offsets = torch.arange(num_graphs, device=z.device) * num_nodes
        edge_index_batched = edge_index.unsqueeze(2) + offsets.view(1, 1, -1)
        edge_index_batched = edge_index_batched.view(2, -1)
        
        x_rec = self.gat(x_gat_in, edge_index_batched)
        x_rec = x_rec.view(batch_size, seq_len, num_nodes, -1)
        return x_rec

class GALMAD_Expert(nn.Module):
    def __init__(self, in_channels=22, hidden_dim=32):
        super().__init__()
        self.encoder = GALMAD_Encoder(in_channels, hidden_dim)
        self.decoder = GALMAD_Decoder(in_channels, hidden_dim)
        
    def forward(self, x, edge_index):
        seq_len = x.shape[1]
        z = self.encoder(x, edge_index)
        x_rec = self.decoder(z, seq_len, edge_index)
        return x_rec

class Router(nn.Module):
    def __init__(self, input_dim=6336, num_experts=5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, num_experts),
            nn.Softmax(dim=-1)
        )
        
    def forward(self, x_flat):
        return self.net(x_flat)

class MoE_GAL_MAD(nn.Module):
    def __init__(self, in_channels=22, hidden_dim=32, num_experts=5):
        super().__init__()
        self.num_experts = num_experts
        self.experts = nn.ModuleList([GALMAD_Expert(in_channels, hidden_dim) for _ in range(num_experts)])
        self.router = Router(input_dim=6336, num_experts=num_experts)
        self.register_buffer('edge_index', EDGE_INDEX)
        
    def forward(self, x):
        batch_size, seq_len, num_nodes, in_channels = x.shape
        x_flat = x.view(batch_size, -1) # Flatten per-window (batch, 6336)
        
        g = self.router(x_flat) # (batch, num_experts)
        
        out = 0
        for i, expert in enumerate(self.experts):
            x_rec_i = expert(x, self.edge_index) # (batch, seq, nodes, features)
            weight = g[:, i].view(batch_size, 1, 1, 1)
            out = out + weight * x_rec_i
            
        return out, g


## 4. Training Loop
Ensure `train.pt` is in the correct path before running.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import time
import pickle
import os

def load_balancing_loss(g):
    num_experts = g.size(1)
    f = g.mean(dim=0) 
    lb_loss = num_experts * torch.sum(f ** 2)
    return lb_loss

def train_model():
    # Giảm batch_size xuống để tránh Out of Memory (OOM) vì 5 GNN chạy song song rất nặng RAM
    batch_size = 128 
    learning_rate = 0.001
    epochs = 20
    alpha = 0.01 
    
    train_path = 'train/train.pt' if os.path.exists('train/train.pt') else 'E:/VHT/AD/train/train.pt'
    
    print("Loading dataset...")
    train_dataset = RSAnomicDataset(
        data_path=train_path, 
        seq_len=24, 
        stride=16,
        is_train=True
    )
    
    train_loader = DataLoader(
        train_dataset, 
        batch_size=batch_size, 
        shuffle=True, 
        num_workers=0,
        drop_last=False
    )
    
    print(f"Dataset loaded. Total samples: {len(train_dataset)}")
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    model = MoE_GAL_MAD(in_channels=22, hidden_dim=32, num_experts=5).to(device)
    
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    
    print("Starting training...")
    epoch_losses = []
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        total_mse = 0
        total_lb = 0
        start_time = time.time()
        
        for batch_idx, batch_x in enumerate(train_loader):
            batch_x = batch_x.to(device)
            
            outputs, g = model(batch_x)
            
            mse = criterion(outputs, batch_x)
            lb = load_balancing_loss(g)
            loss = mse + alpha * lb
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            total_loss += loss.item()
            total_mse += mse.item()
            total_lb += lb.item()
            
            if (batch_idx + 1) % 10 == 0:
                print(f"Epoch [{epoch+1}/{epochs}], Step [{batch_idx+1}/{len(train_loader)}], Total: {loss.item():.4f} (MSE: {mse.item():.4f}, LB: {lb.item():.4f})")
                
        scheduler.step()
        epoch_time = time.time() - start_time
        print(f"Epoch [{epoch+1}/{epochs}] completed in {epoch_time:.2f}s, Avg Total: {total_loss/len(train_loader):.4f}, LR: {scheduler.get_last_lr()[0]:.6f}")
        epoch_losses.append(total_loss/len(train_loader))
        
    print("Training finished.")
    
    print("Saving model and scaler...")
    torch.save(model.state_dict(), 'moe_gal_mad_model.pth')
    
    with open('scaler.pkl', 'wb') as f:
        pickle.dump(train_dataset.scaler, f)
    print("Model and scaler saved successfully.")
    return epoch_losses

if __name__ == '__main__':
    import multiprocessing
    multiprocessing.freeze_support()
    losses = train_model()

    import matplotlib.pyplot as plt
    plt.figure(figsize=(10, 5))
    plt.plot(range(1, len(losses) + 1), losses, marker='o', linestyle='-', color='b')
    plt.title('Training Loss over Epochs')
    plt.xlabel('Epoch')
    plt.ylabel('Average Total Loss')
    plt.grid(True)
    plt.show()



## 5. Inference and SHAP Explanation
Ensure `test5.pt` and `train.pt` are in the correct paths.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import shap
import pickle
import json
import os
from torch.utils.data import DataLoader

class LossWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        
    def forward(self, x):
        x_rec, _ = self.model(x)
        mse_per_sample = torch.mean((x_rec - x) ** 2, dim=(1, 2, 3))
        return mse_per_sample.unsqueeze(1)

def detect_and_explain():
    seq_len = 24
    c_threshold = 2.0
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    model = MoE_GAL_MAD(in_channels=22, hidden_dim=32, num_experts=5)
    model.load_state_dict(torch.load('moe_gal_mad_model.pth', map_location=device))
    model.to(device)
    model.eval()
    
    with open('scaler.pkl', 'rb') as f:
        scaler = pickle.load(f)
        
    print("Loading test data...")
    test_path = 'test/test5.pt' if os.path.exists('test/test5.pt') else 'E:/VHT/AD/test/test5.pt'
    test_dataset = RSAnomicDataset(
        data_path=test_path, 
        seq_len=seq_len,
        stride=16,
        is_train=False,
        scaler=scaler
    )
    
    print("Loading background data for SHAP...")
    train_path = 'train/train.pt' if os.path.exists('train/train.pt') else 'E:/VHT/AD/train/train.pt'
    train_dataset = RSAnomicDataset(
        data_path=train_path, 
        seq_len=seq_len,
        stride=16,
        is_train=True,
        scaler=scaler
    )
    
    bg_indices = np.random.choice(len(train_dataset), 50, replace=False) # Reduced to 50 to save RAM during SHAP
    background_data = torch.stack([train_dataset[i] for i in bg_indices]).to(device)
    
    loss_model = LossWrapper(model).to(device)
    loss_model.eval()
    
    print("Initializing SHAP GradientExplainer...")
    explainer = shap.GradientExplainer(loss_model, background_data)
    
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)
    
    detected_anomalies = 0
    results_json = []
    tp = 0; fp = 0; tn = 0; fn = 0
    
    print("Starting inference and explanation...")
    for idx, (x, y) in enumerate(test_loader):
        x = x.to(device)
        
        with torch.no_grad():
            x_rec, g = model(x)
            mse_loss = torch.mean((x_rec - x) ** 2).item()
            anomaly_score = 1 / (1 + np.exp(-(mse_loss - c_threshold)))
            
            # Since batch_size=1, g is (1, 5)
            expert_dist = g[0].tolist()
            expert_dist_rounded = [round(val, 4) for val in expert_dist]
            
        is_pred_anomaly = anomaly_score > 0.5
        is_true_anomaly = y.sum().item() > 0
        if is_pred_anomaly and is_true_anomaly: tp += 1
        elif is_pred_anomaly and not is_true_anomaly: fp += 1
        elif not is_pred_anomaly and not is_true_anomaly: tn += 1
        else: fn += 1

        if is_pred_anomaly:
            detected_anomalies += 1
            print(f"MSE Loss: {mse_loss:.4f}, Anomaly Score: {anomaly_score:.4f}, Ground Truth Label: {y.tolist()}")
            
            x.requires_grad_(True)
            torch.backends.cudnn.enabled = False
            shap_values = explainer.shap_values(x)
            torch.backends.cudnn.enabled = True
            
            if isinstance(shap_values, list):
                shap_vals = shap_values[0]
            else:
                shap_vals = shap_values
                
            shap_vals = np.array(shap_vals)[0]
            abs_shap = np.abs(shap_vals)
            time_aggregated_shap = np.sum(abs_shap, axis=0)
            
            time_aggregated_shap[:, 19] *= 4
            time_aggregated_shap[:, 0] *= 4
            
            service_scores = np.sum(time_aggregated_shap, axis=1)
            anomalous_service_idx = np.argmax(service_scores)
            
            containers = ['payment', 'shipping', 'redis', 'mongo', 'dispatch', 'rabbitmq', 'user', 'mysql', 'catalogue', 'ratings', 'web', 'cart']
            anomalous_service_name = containers[anomalous_service_idx]
            
            print(f">> Localized Anomalous Service: {anomalous_service_name} (Index: {anomalous_service_idx})")
            
            feature_scores = time_aggregated_shap[anomalous_service_idx]
            root_cause_metric_idx = np.argmax(feature_scores)
            
            cumulative_cols = ['container_cpu_system_seconds_total', 'container_cpu_usage_seconds_total', 'container_cpu_user_seconds_total', 'container_network_receive_bytes_total', 'container_network_receive_errors_total', 'container_network_receive_packets_dropped_total', 'container_network_receive_packets_total', 'container_network_transmit_bytes_total', 'container_network_transmit_errors_total', 'container_network_transmit_packets_dropped_total', 'container_network_transmit_packets_total', 'container_fs_io_time_seconds_total', 'container_memory_failures_total', 'container_memory_failcnt', 'container_fs_write_seconds_total']
            other_cols = ['container_fs_usage_bytes', 'container_memory_rss', 'container_memory_usage_bytes', 'container_memory_working_set_bytes']
            cols = other_cols + cumulative_cols
            
            if root_cause_metric_idx < len(cols):
                root_cause_metric_name = cols[root_cause_metric_idx]
            elif root_cause_metric_idx == 19: root_cause_metric_name = "response_time_sum"
            elif root_cause_metric_idx == 20: root_cause_metric_name = "response_time_ma_24"
            elif root_cause_metric_idx == 21: root_cause_metric_name = "response_time_ma_240"
            else: root_cause_metric_name = "Unknown"
                
            print(f">> Localized Root Cause Metric: {root_cause_metric_name} (Index: {root_cause_metric_idx})")
            
            results_json.append({
                "window_idx": idx,
                "mse_loss": float(mse_loss),
                "anomaly_score": float(anomaly_score),
                "expert_distribution": expert_dist_rounded,
                "anomalous_service": anomalous_service_name,
                "anomalous_service_idx": int(anomalous_service_idx),
                "root_cause_metric": root_cause_metric_name,
                "root_cause_metric_idx": int(root_cause_metric_idx)
            })
            
    accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

    metrics = {
        "accuracy": float(accuracy),
        "recall": float(recall),
        "specificity": float(specificity),
        "tp": tp, "fp": fp, "tn": tn, "fn": fn
    }
    print(f"\nTotal Anomalies Detected: {detected_anomalies}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"Specificity: {specificity:.4f}")
    
    with open('anomalies_moe_galmad_report.json', 'w', encoding='utf-8') as f:
        final_output = {'evaluation_metrics': metrics, 'detected_anomalies': results_json}
        json.dump(final_output, f, indent=4, ensure_ascii=False)
    print("Saved anomaly results to anomalies_moe_galmad_report.json")

if __name__ == '__main__':
    detect_and_explain()


## 6. Hyperparameter Tuning (PR Curve)
Quét thử nghiệm ngưỡng `c_threshold` từ 0.5 đến 4.0 để vẽ đường cong Precision-Recall, giúp tìm ra điểm giới hạn tối ưu nhất.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
import pickle
import os

def plot_pr_curve():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # 1. Tải mô hình
    model = MoE_GAL_MAD(in_channels=22, hidden_dim=32, num_experts=5)
    model.load_state_dict(torch.load('moe_gal_mad_model.pth', map_location=device))
    model.to(device)
    model.eval()
    
    # 2. Tải dữ liệu test
    with open('scaler.pkl', 'rb') as f:
        scaler = pickle.load(f)
        
    test_path = 'test/test5.pt' if os.path.exists('test/test5.pt') else 'E:/VHT/AD/test/test5.pt'
    try:
        test_dataset = RSAnomicDataset(data_path=test_path, seq_len=24, stride=16, is_train=False, scaler=scaler)
    except FileNotFoundError:
        print("Vui lòng kiểm tra lại đường dẫn file test5.pt")
        return
        
    test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)
    
    all_mse_loss = []
    all_true_labels = []
    
    print("Đang chạy Inference toàn bộ tập test để trích xuất MSE Loss...")
    with torch.no_grad():
        for x, y in test_loader:
            x = x.to(device)
            x_rec = model(x)
            
            if isinstance(x_rec, tuple):
                x_rec = x_rec[0]
                
            mse_per_sample = torch.mean((x_rec - x) ** 2, dim=(1,2,3))
            all_mse_loss.extend(mse_per_sample.cpu().numpy())
            
            true_labels = (y.sum(dim=1) > 0).int() if y.dim() > 1 else (y > 0).int()
            all_true_labels.extend(true_labels.cpu().numpy())
            
    all_mse_loss = np.array(all_mse_loss)
    all_true_labels = np.array(all_true_labels)
    
    # 3. Quét c_threshold từ 0.5 đến 4.0
    thresholds = np.arange(0.5, 4.5, 0.5)
    precisions = []
    recalls = []
    f1_scores = []
    
    print(f"\n{'c_thresh':<10} | {'Precision':<10} | {'Recall':<10} | {'F1-Score':<10}")
    print("-" * 50)
    
    best_c = thresholds[0]
    best_f1 = 0
    
    for c in thresholds:
        preds = (all_mse_loss > c).astype(int)
        
        tp = np.sum((preds == 1) & (all_true_labels == 1))
        fp = np.sum((preds == 1) & (all_true_labels == 0))
        fn = np.sum((preds == 0) & (all_true_labels == 1))
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        
        precisions.append(precision)
        recalls.append(recall)
        f1_scores.append(f1)
        
        if f1 > best_f1:
            best_f1 = f1
            best_c = c
            
        print(f"{c:<10.1f} | {precision:<10.4f} | {recall:<10.4f} | {f1:<10.4f}")
        
    print(f"\n=> Threshold TỐT NHẤT dựa trên F1-Score: c = {best_c:.1f} (F1 = {best_f1:.4f})")
        
    # 4. Vẽ đồ thị PR Curve
    plt.figure(figsize=(10, 6))
    plt.plot(recalls, precisions, marker='o', linestyle='-', color='b', linewidth=2, markersize=8)
    
    for i, c in enumerate(thresholds):
        plt.annotate(f"c={c:.1f}", (recalls[i], precisions[i]), textcoords="offset points", xytext=(0,10), ha='center')
        
    plt.title('Precision-Recall Curve (Tuning c_threshold)')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.xlim(0, 1.05)
    plt.ylim(0, 1.05)
    plt.show()

if __name__ == '__main__':
    plot_pr_curve()


# MỚI: Tự động rà quét Grid Search tìm ngưỡng c tối ưu
Khối code dưới đây tự động tìm ngưỡng c tốt nhất để đạt F1-Score cao nhất.

In [ ]:
import numpy as np
import torch
import pickle
import os
import warnings
from torch.utils.data import DataLoader

def evaluate_model_grid_search():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = MoE_GAL_MAD(in_channels=22, hidden_dim=32, num_experts=5)
    model.load_state_dict(torch.load('moe_gal_mad_model.pth', map_location=device))
    model.to(device)
    model.eval()
    
    with open('scaler.pkl', 'rb') as f:
        scaler = pickle.load(f)
        
    test_path = 'test/test5.pt' if os.path.exists('test/test5.pt') else 'E:/VHT/AD/test/test5.pt'
    test_dataset = RSAnomicDataset(
        data_path=test_path, 
        seq_len=24, 
        stride=16,
        is_train=False,
        scaler=scaler
    )
    test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)
    
    print("Running inference to collect losses...")
    all_losses = []
    all_labels = []
    
    with torch.no_grad():
        for x, y in test_loader:
            x = x.to(device)
            y_window = y.max(dim=1)[0].numpy()
            
            x_rec, _ = model(x)
            mse = torch.mean((x_rec - x)**2, dim=(1,2,3)).cpu().numpy()
            
            all_losses.extend(mse)
            all_labels.extend(y_window)
            
    all_losses = np.array(all_losses)
    all_labels = np.array(all_labels)
    
    print("Quét ngưỡng C (Grid Search Thresholding)...")
    best_f1 = 0
    best_c = 0
    best_metrics = {}
    
    min_loss, max_loss = np.min(all_losses), np.max(all_losses)
    c_thresholds = np.linspace(min_loss, max_loss, 100)
    
    warnings.filterwarnings('ignore', category=RuntimeWarning)
    
    for c in c_thresholds:
        anomaly_score = 1 / (1 + np.exp(-np.clip(all_losses - c, -700, 700)))
        preds = (anomaly_score > 0.5).astype(int)
        
        tp = np.sum((preds == 1) & (all_labels == 1))
        fp = np.sum((preds == 1) & (all_labels == 0))
        tn = np.sum((preds == 0) & (all_labels == 0))
        fn = np.sum((preds == 0) & (all_labels == 1))
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        
        if f1 > best_f1:
            best_f1 = f1
            best_c = c
            best_metrics = {
                'Threshold (c)': c,
                'Precision': precision,
                'Recall': recall,
                'F1-Score': f1,
                'Accuracy': (tp + tn) / len(all_labels)
            }
            
    print("\n=== KẾT QUẢ TỐT NHẤT MỚI (GRID SEARCH) ===")
    for k, v in best_metrics.items():
        print(f"{k}: {v:.4f}")
        
    return best_metrics

# Bỏ comment dòng dưới để chạy Grid Search
# evaluate_model_grid_search()

